# CLEAR post-video analysis: compare runs

This notebook takes the system's analysis as data and compares runs side by side.
A run is one analysis of one recording, or of one segment of it.

**Getting data in**

1. Open a finished game in **Dev tools → Games → *game***, mark segments in **Recording preview** and analyse them.
2. Click **Export selected** or **Export all**. You get a ZIP containing `manifest.json`, `runs.csv`, and one folder per run with
   `summary.json`, `bounces.csv`, `tracks.csv` (one row per frame), `faults.csv` and the court chart.
3. Drop the ZIPs, unzipped or not, into `notebooks/exports/` and run all cells.

Plain pipeline output folders also work, for example `results/eval/final` or `scripts/infer.py --out …`.
Add them to `EXTRA_SOURCES`.

**What to compare.** Pairwise comparisons (sections 6–8) only make sense for runs of the **same footage**.
Examples: the same segment analysed before and after a model change, with different calibrations, or a segment
against the full-recording run that contains it. Frame numbers in every export are absolute to the source
recording, so the runs line up.

In [ ]:
from pathlib import Path
import itertools
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NB_DIR = Path.cwd() if (Path.cwd() / "clear_runs.py").exists() else Path.cwd() / "notebooks"
REPO = NB_DIR.parent
sys.path.insert(0, str(NB_DIR))
import clear_runs as cr

EXPORT_DIR = NB_DIR / "exports"          # drop exported ZIPs / folders here
EXTRA_SOURCES = [                         # any other run folders or bundles
    REPO / "results/eval/baseline",
    REPO / "results/eval/final",
]
INCLUDE_EXTRA = True                      # set False to only use exports/
BOUNCE_MATCH_TOL_S = 0.15                 # bounces within this time of each other are "the same bounce"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Fixed categorical order: a run keeps its colour in every chart.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CALL_STYLE = {  # colour + marker, so calls never rely on colour alone
    "IN": dict(color="#008300", marker="o", label="In"),
    "OUT": dict(color="#e34948", marker="X", label="Out"),
    "CLOSE": dict(color="#eda100", marker="^", label="Close call (doubt zone)"),
}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#9a9a95", "axes.labelcolor": "#3d3d3a", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "lines.linewidth": 2, "font.size": 10,
})

In [ ]:
sources = cr.find_sources(EXPORT_DIR) + ([p for p in EXTRA_SOURCES if Path(p).exists()] if INCLUDE_EXTRA else [])
if not sources:
    raise SystemExit(f"No runs found. Put export ZIPs in {EXPORT_DIR} or set EXTRA_SOURCES.")
data = cr.load_runs(sources)
RUN_IDS = data.ids()
COLOR = {rid: SERIES[i] if i < len(SERIES) else "#8a8a85" for i, rid in enumerate(RUN_IDS)}
if len(RUN_IDS) > len(SERIES):
    print(f"Note: {len(RUN_IDS)} runs; runs past the 8th are drawn grey. Facet or filter for colour identity.")
print(f"Loaded {len(RUN_IDS)} runs from {len(sources)} source(s):")
for s in sources:
    print("  ", s)
print(f"bounces: {len(data.bounces)} rows · tracks: {len(data.tracks)} rows · faults: {len(data.faults)} rows")

## 1. Overview
One row per run. `start_s`/`end_s` are positions in the source recording.

In [ ]:
overview = cr.summary_table(data)
display(overview)

## 2. Headline metrics side by side
Each panel has its own axis. Metrics with different units are never shared on one scale.

In [ ]:
metrics = [
    ("ball_coverage_pct", "Ball tracked (% of frames)"),
    ("avg_speed_kmh", "Avg ball speed (km/h)"),
    ("max_speed_kmh", "Top ball speed (km/h)"),
    ("bounces", "Bounces judged"),
    ("doubt_rate_pct", "Close calls (% of bounces)"),
    ("rally_strokes", "Strokes"),
]
labels = [data.runs.set_index("run_id").loc[r, "label"] for r in RUN_IDS]
fig, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)
for ax, (col, title) in zip(axes.flat, metrics):
    vals = data.runs.set_index("run_id").loc[RUN_IDS, col].astype(float).fillna(0)
    bars = ax.barh(range(len(RUN_IDS)), vals, color=[COLOR[r] for r in RUN_IDS], height=0.6)
    ax.set_yticks(range(len(RUN_IDS)), labels)
    ax.invert_yaxis()
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.grid(axis="y", visible=False)
    ax.bar_label(bars, fmt="%g", padding=3, fontsize=8, color="#3d3d3a")
    ax.margins(x=0.15)
plt.show()

## 3. Ball speed distribution
Uses per-frame `tracks.csv` (detected frames only) when available. Older runs without tracks fall back to bounce speeds.

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(RUN_IDS)), 4.5), constrained_layout=True)
series, used = [], []
for rid in RUN_IDS:
    t = data.tracks[data.tracks["run_id"] == rid] if not data.tracks.empty else pd.DataFrame()
    if len(t):
        s = t.loc[(t["ball_state"] == "detected") & (t["ball_speed_kmh"] > 5), "ball_speed_kmh"]
        src = "tracks"
    else:
        b = data.bounces[data.bounces["run_id"] == rid] if not data.bounces.empty else pd.DataFrame()
        s = b["speed_kmh"] if len(b) else pd.Series(dtype=float)
        src = "bounces"
    if len(s):
        series.append(s.to_numpy()); used.append((rid, src))
if series:
    bp = ax.boxplot(series, patch_artist=True, widths=0.5,
                    medianprops=dict(color="#0b0b0b", linewidth=1.5), flierprops=dict(markersize=3, alpha=.5))
    for patch, (rid, _) in zip(bp["boxes"], used):
        patch.set_facecolor(COLOR[rid]); patch.set_alpha(.75); patch.set_edgecolor(COLOR[rid])
    ax.set_xticks(range(1, len(used) + 1), [f"{data.runs.set_index('run_id').loc[r, 'label']}\n({src})" for r, src in used],
                  fontsize=8)
    ax.set_ylabel("Ball speed (km/h)")
    ax.grid(axis="x", visible=False)
    ax.set_title("Ball speed per run", loc="left", fontweight="bold")
    plt.show()
else:
    print("No speed data in these runs.")

## 4. Where bounces land
One court per run (small multiples). The far end is at the top. Marker shape and colour both encode the call.

In [ ]:
def call_kind(row):
    return "CLOSE" if row.get("in_doubt_zone") else ("IN" if row.get("is_in") else "OUT")

n = len(RUN_IDS)
cols = min(4, n)
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 5.6 * rows), squeeze=False, constrained_layout=True)
for ax, rid in zip(axes.flat, RUN_IDS):
    cr.draw_court(ax)
    b = data.bounces[data.bounces["run_id"] == rid] if not data.bounces.empty else pd.DataFrame()
    if len(b):
        kinds = b.apply(call_kind, axis=1)
        for k, st in CALL_STYLE.items():
            sel = b[kinds == k]
            if len(sel):
                ax.scatter(sel["x_m"], sel["y_m"], s=64, color=st["color"], marker=st["marker"],
                           edgecolors="white", linewidths=1.2, label=st["label"], zorder=3)
    ax.set_title(f"{data.runs.set_index('run_id').loc[rid, 'label']}\n{len(b)} bounces", fontsize=9)
for ax in list(axes.flat)[n:]:
    ax.set_visible(False)
handles = [plt.Line2D([], [], linestyle="", markersize=8, markeredgecolor="white", markerfacecolor=s["color"],
                      marker=s["marker"], label=s["label"]) for s in CALL_STYLE.values()]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.show()

## 5. Calls and line margins
Left: call split per run. Right: signed distance from the closest line (negative = out), with the ±15 cm doubt zone shaded.

In [ ]:
if not data.bounces.empty:
    k = data.bounces.assign(kind=data.bounces.apply(call_kind, axis=1))
    counts = k.pivot_table(index="run_id", columns="kind", values="frame_idx", aggfunc="count", fill_value=0)
    counts = counts.reindex(index=[r for r in RUN_IDS if r in counts.index], columns=list(CALL_STYLE), fill_value=0)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 0.55 * len(counts) + 2), constrained_layout=True,
                                 gridspec_kw=dict(width_ratios=[1, 1.4]))
    left = np.zeros(len(counts))
    for kind, st in CALL_STYLE.items():
        a1.barh(range(len(counts)), counts[kind], left=left, color=st["color"], height=0.6,
                edgecolor="white", linewidth=2, label=st["label"])
        left += counts[kind].to_numpy()
    a1.set_yticks(range(len(counts)), [data.runs.set_index("run_id").loc[r, "label"] for r in counts.index])
    a1.invert_yaxis(); a1.grid(axis="y", visible=False); a1.set_xlabel("Bounces")
    a1.legend(frameon=False, loc="upper center", bbox_to_anchor=(.5, -.12 - 1.2 / (len(counts) + 3)), ncol=3, fontsize=8)
    a1.set_title("Calls per run", loc="left", fontweight="bold")

    a2.axvspan(-15, 15, color="#eda100", alpha=.12, lw=0)
    a2.axvline(0, color="#52514e", lw=1)
    for i, rid in enumerate(counts.index):
        m = k.loc[k["run_id"] == rid, "signed_margin_cm"].dropna()
        jitter = np.random.default_rng(0).uniform(-.18, .18, len(m))
        a2.scatter(m, i + jitter, s=22, color=COLOR[rid], alpha=.8, edgecolors="white", linewidths=.6)
    a2.set_yticks(range(len(counts)), [data.runs.set_index("run_id").loc[r, "label"] for r in counts.index])
    a2.invert_yaxis(); a2.grid(axis="y", visible=False)
    a2.set_xlabel("Signed margin to closest line (cm); positive = in")
    a2.set_title("How close each call was", loc="left", fontweight="bold")
    plt.show()
else:
    print("No bounces in these runs.")

## 6. Head-to-head: run A vs run B

Pick two runs of **the same footage**. Bounces are matched one-to-one by time: closest pairs first, within
`BOUNCE_MATCH_TOL_S`. With A as the reference:
- **recall** is the share of A's bounces that B also found.
- **precision** is the share of B's bounces that A also has.
- **call agreement** is computed over matched pairs.

In [ ]:
# Defaults: the first two runs. Change to any ids from the overview table.
RUN_A = RUN_IDS[0]
RUN_B = RUN_IDS[1] if len(RUN_IDS) > 1 else RUN_IDS[0]

def run_bounces(rid, lo=-np.inf, hi=np.inf):
    if data.bounces.empty:
        return pd.DataFrame(columns=["t_s"])
    b = data.bounces[data.bounces["run_id"] == rid]
    return b[(b["t_s"] >= lo) & (b["t_s"] < hi)]

# Only the time window both runs cover counts; a segment never "misses" bounces outside itself.
_ra, _rb = data.runs.set_index("run_id").loc[RUN_A], data.runs.set_index("run_id").loc[RUN_B]
WIN = (max(_ra["start_s"], _rb["start_s"]), min(_ra["end_s"], _rb["end_s"]))
matched = cr.match_bounces(run_bounces(RUN_A, *WIN), run_bounces(RUN_B, *WIN), tol_s=BOUNCE_MATCH_TOL_S)
print(f"A = {RUN_A}\nB = {RUN_B}\nShared window: {WIN[0]:.2f}–{WIN[1]:.2f} s\n")
if WIN[1] <= WIN[0]:
    print("These runs do not overlap in time: pick two runs of the same footage.")
display(cr.bounce_agreement(matched).to_frame("A vs B"))

if not matched.empty:
    issues = matched[(matched["status"] != "matched") | (matched["call_agrees"] == False)]  # noqa: E712
    print(f"{len(issues)} bounce(s) missing from one run or called differently:")
    display(issues[["status", "t_s_a", "t_s_b", "call_a", "call_b", "signed_margin_cm_a", "signed_margin_cm_b",
                    "pos_diff_cm", "dt_ms"]].round(2))

In [ ]:
if not matched.empty:
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 6.2), constrained_layout=True, gridspec_kw=dict(width_ratios=[1, 1.6]))
    cr.draw_court(ax)
    m = matched[matched["status"] == "matched"]
    for _, r in m.iterrows():
        ax.plot([r["x_m_a"], r["x_m_b"]], [r["y_m_a"], r["y_m_b"]], color="#9a9a95", lw=1, zorder=2)
    # A filled, B hollow and larger: identical positions stay visible as a ringed dot.
    pts = matched[matched["x_m_a"].notna()]
    ax.scatter(pts["x_m_a"], pts["y_m_a"], s=44, marker="o", color=COLOR[RUN_A],
               edgecolors="white", linewidths=1.2, zorder=3, label=f"A: {RUN_A}")
    pts = matched[matched["x_m_b"].notna()]
    ax.scatter(pts["x_m_b"], pts["y_m_b"], s=150, marker="o", facecolors="none", edgecolors=COLOR[RUN_B],
               linewidths=2, zorder=4, label=f"B: {RUN_B}")
    ax.legend(loc="lower center", bbox_to_anchor=(.5, -.12), frameon=False, fontsize=8)
    ax.set_title("Bounce positions (grey line joins a matched pair)", fontsize=9)

    for y, (side, rid) in enumerate((("a", RUN_A), ("b", RUN_B))):
        pts = matched[matched[f"t_s_{side}"].notna()]
        style = (dict(color=COLOR[rid], edgecolors="white", linewidths=1.2, s=60) if side == "a"
                 else dict(facecolors="none", edgecolors=COLOR[rid], linewidths=2, s=90))
        ax2.scatter(pts[f"t_s_{side}"], np.full(len(pts), y), marker="o", zorder=3, **style)
        for _, r in pts.iterrows():
            ax2.annotate(str(r[f"call_{side}"]), (r[f"t_s_{side}"], y), xytext=(0, 9), textcoords="offset points",
                         ha="center", fontsize=7, color="#3d3d3a")
    for _, r in m.iterrows():
        ax2.plot([r["t_s_a"], r["t_s_b"]], [0, 1], color="#c3c2b7", lw=1, zorder=1)
    ax2.set_yticks([0, 1], [f"A  {RUN_A}", f"B  {RUN_B}"]); ax2.set_ylim(-.6, 1.6)
    ax2.set_xlabel("Time in recording (s)"); ax2.grid(axis="y", visible=False)
    ax2.set_title("Bounce timeline", loc="left", fontweight="bold")
    plt.show()

## 7. Frame-by-frame ball tracking (A vs B)
Needs `tracks.csv` in both runs, which every export from the dev page has. Frames are joined on their absolute frame number.

In [ ]:
joined, track_stats = cr.compare_tracks(data.tracks, RUN_A, RUN_B)
if joined.empty:
    print("Skipped: one or both runs have no tracks.csv (older runs). Re-export from the dev page to get per-frame data.")
else:
    display(track_stats.to_frame("A vs B"))
    fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True, constrained_layout=True)
    for side, rid in (("a", RUN_A), ("b", RUN_B)):
        s = joined[f"ball_speed_kmh_{side}"].where(joined[f"ball_state_{side}"] == "detected")
        axes[0].plot(joined[f"t_s_{side}"], s, color=COLOR[rid], label=rid)
    axes[0].set_ylabel("Ball speed (km/h)"); axes[0].legend(frameon=False, fontsize=8)
    axes[0].set_title("Ball speed over time (detected frames)", loc="left", fontweight="bold")
    both = joined[(joined["ball_state_a"] == "detected") & (joined["ball_state_b"] == "detected")]
    d = np.hypot(both["ball_x_m_b"] - both["ball_x_m_a"], both["ball_y_m_b"] - both["ball_y_m_a"]) * 100
    axes[1].plot(both["t_s_a"], d, color="#52514e", lw=1.2)
    axes[1].set_ylabel("Position gap (cm)"); axes[1].set_xlabel("Time in recording (s)")
    axes[1].set_title("Distance between A's and B's ball position on court", loc="left", fontweight="bold")
    plt.show()

## 8. Every comparable pair

All pairs of runs whose time ranges overlap on the same camera end. Runs without an end, such as plain result
folders, are compared with each other. Use this to spot which change moved the numbers most.

In [ ]:
def comparable(a, b):
    ra, rb = data.runs.set_index("run_id").loc[a], data.runs.set_index("run_id").loc[b]
    same_end = (ra["end"] == rb["end"]) or (pd.isna(ra["end"]) and pd.isna(rb["end"]))
    same_game = (ra["game_id"] == rb["game_id"]) or (pd.isna(ra["game_id"]) and pd.isna(rb["game_id"]))
    overlap = min(ra["end_s"], rb["end_s"]) > max(ra["start_s"], rb["start_s"])
    return bool(same_end and same_game and overlap)

pair_rows = []
for a, b in itertools.combinations(RUN_IDS, 2):
    if not comparable(a, b):
        continue
    ba = data.bounces[data.bounces["run_id"] == a] if not data.bounces.empty else pd.DataFrame(columns=["t_s"])
    bb = data.bounces[data.bounces["run_id"] == b] if not data.bounces.empty else pd.DataFrame(columns=["t_s"])
    # Only bounces inside the shared time window count.
    ra, rb = data.runs.set_index("run_id").loc[a], data.runs.set_index("run_id").loc[b]
    lo, hi = max(ra["start_s"], rb["start_s"]), min(ra["end_s"], rb["end_s"])
    win = lambda df: df[(df["t_s"] >= lo) & (df["t_s"] < hi)] if len(df) else df
    agg = cr.bounce_agreement(cr.match_bounces(win(ba), win(bb), BOUNCE_MATCH_TOL_S))
    _, ts = cr.compare_tracks(data.tracks, a, b)
    pair_rows.append({"run_a": a, "run_b": b, "window_s": f"{lo:.2f}–{hi:.2f}", **agg.to_dict(),
                      "rmse_ball_pos_cm": ts.get("rmse_ball_pos_cm", np.nan)})
pairwise = pd.DataFrame(pair_rows)
display(pairwise if len(pairwise) else "No overlapping runs of the same footage.")

## 9. Save the comparison
Writes CSVs to `notebooks/outputs/<timestamp>/` for use in other tools.

In [ ]:
stamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
OUT = NB_DIR / "outputs" / stamp
OUT.mkdir(parents=True, exist_ok=True)
data.runs.to_csv(OUT / "runs.csv", index=False)
data.bounces.to_csv(OUT / "bounces.csv", index=False)
data.faults.to_csv(OUT / "faults.csv", index=False)
if not matched.empty:
    matched.to_csv(OUT / f"bounce_match_A_vs_B.csv", index=False)
if len(pairwise):
    pairwise.to_csv(OUT / "pairwise.csv", index=False)
print("Saved to", OUT.relative_to(REPO))